In [5]:
import datasets
import unet
from SplitNet import SplitNet
from training import darcy_loss

import torch
from torch import nn
from torch.optim import Adam
import numpy as np
from tqdm import tqdm
import os

torch.random.manual_seed(42)
np.random.seed(42)

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
def just_darcy(out) -> torch.Tensor:

    # If we assume the output is in order k,pres,phi
    # pres_grad is the gradient of the pressure along the y and x directions as a tuple
    pres_grad = torch.gradient(out[:, 1:2], dim=(-2,-1))

    # get velocity by multiplying the gradient by the conductivity
    y_grad = pres_grad[0] * out[:, 0:1]
    x_grad = pres_grad[1] * out[:, 0:1]

    # compute the divergence by the second derivative of the gradients and adding them together
    yy_grad = torch.gradient(y_grad, spacing=(1,),dim=(-2,))[0]
    xx_grad = torch.gradient(x_grad, spacing=(1,),dim=(-1,))[0]
    final = yy_grad + xx_grad

    # total divergence should be 0
    loss = (final**2)

    return loss


def make_model(model_type="splitnet_attn"):
    model_type = model_type.lower()

    if model_type == "splitnet_attn":
        return SplitNet(attn=True).to(device)
    elif model_type == "splitnet":
        return SplitNet(attn=False).to(device)
    elif model_type == "unet":
        return unet.SmallUnet(channels=2).to(device)
    elif model_type == "attn_unet":
        return unet.AttnUnet(channels=2).to(device)
    else:
        raise ValueError(f"Wrong model type: {model_type}")


def make_loaders(train_sims, val_sims, dataset_mode="border", batch_size=8):
    if dataset_mode == "border":
        train_data = datasets.BorderThinDatasetLimited(train_sims, channels="KP")
        val_data   = datasets.BorderThinDatasetLimited(val_sims, channels="KP")

    elif dataset_mode == "fixed":
        train_data = datasets.FixedThinDatasetLimited(train_sims, channels="KP")
        val_data   = datasets.FixedThinDatasetLimited(val_sims, channels="KP")

    elif dataset_mode == "random":
        train_data = datasets.RandomDenseDatasetLimited(train_sims, channels="KP")
        val_data   = datasets.RandomDenseDatasetLimited(val_sims, channels="KP")

    else:
        raise ValueError("dataset_mode must be 'border', 'fixed', or 'random'")

    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader   = torch.utils.data.DataLoader(val_data, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader


def expand_mask_for_channels(mask, out):
    return mask.expand(-1, out.shape[1], -1, -1)


def mse_on_region(pred, target, mask):
    mask = mask.float()
    diff2 = ((pred - target) ** 2) * mask
    denom = mask.sum()
    if denom.item() == 0:
        return torch.tensor(0.0, device=pred.device)
    return diff2.sum() / denom

In [7]:



def evaluate_loader(model, loader, crit):
    model.eval()

    total_mse = 0.0
    mask_mse = 0.0
    nonmask_mse = 0.0
    total_darcy = 0.0
    n_batches = 0

    with torch.no_grad():
        for feat, label, mask in loader:
            feat = feat.to(device)
            label = label.to(device)

            if label.shape[1] != feat.shape[1]:
                label = label[:, :feat.shape[1]]

            if mask.dim() == 3:
                mask = mask.unsqueeze(1)
            mask = mask.to(device).bool()

            _, out = darcy_loss(model, feat)

            if label.shape[1] != out.shape[1]:
                label = label[:, :out.shape[1]]

            mask_c = expand_mask_for_channels(mask, out)
            nonmask_c = ~mask_c

            total_mse += crit(out, label).item()
            mask_mse += mse_on_region(out, label, mask_c).item()
            nonmask_mse += mse_on_region(out, label, nonmask_c).item()
            total_darcy += just_darcy(out).mean().item()
            n_batches += 1

    return {
        "total_mse": total_mse / n_batches,
        "mask_mse": mask_mse / n_batches,
        "nonmask_mse": nonmask_mse / n_batches,
        "darcy": total_darcy / n_batches,
    }


def run_experiment(
    model_type="splitnet_attn",
    dataset_mode="border",
    loss_weights=(1, 1),
    epochs=250,
    batch_size=8,
    lr=1e-3,
    save_prefix=None
):
    train_sims = np.load("../train_sims.npy")
    train_sims = train_sims[train_sims < 500]

    val_sims = np.load("../val_sims.npy")
    val_sims = val_sims[val_sims < 500]

    model = make_model(model_type)
    optim = Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()

    train_loader, val_loader = make_loaders(train_sims,val_sims,dataset_mode=dataset_mode,batch_size=batch_size)

    history = {
        "train_loss_used": [],
        "train_total": [],
        "train_mask": [],
        "train_nonmask": [],
        "train_darcy": [],
        "val_total": [],
        "val_mask": [],
        "val_nonmask": [],
        "val_darcy": [],
    }

    best_val_loss = float("inf")
    best_epoch = 0

    for epoch in tqdm(range(1, epochs + 1)):
        model.train()

        epoch_loss = 0.0
        n_batches = 0

        for feat, label, mask in train_loader:
            optim.zero_grad()

            feat = feat.to(device)
            label = label.to(device)

            if label.shape[1] != feat.shape[1]:
                label = label[:, :feat.shape[1]]

            if mask.dim() == 3:
                mask = mask.unsqueeze(1)
            mask = mask.to(device).bool()

            p_loss, out = darcy_loss(model, feat)

            if label.shape[1] != out.shape[1]:
                label = label[:, :out.shape[1]]

            mask_c = expand_mask_for_channels(mask, out)

            loss = crit(out * mask_c.float(), label * mask_c.float()) * loss_weights[0] + p_loss * loss_weights[1]
            loss.backward()
            optim.step()

            epoch_loss += loss.item()
            n_batches += 1

        history["train_loss_used"].append(epoch_loss / n_batches)

        train_metrics = evaluate_loader(model, train_loader, crit)
        val_metrics = evaluate_loader(model, val_loader, crit)

        history["train_total"].append(train_metrics["total_mse"])
        history["train_mask"].append(train_metrics["mask_mse"])
        history["train_nonmask"].append(train_metrics["nonmask_mse"])
        history["train_darcy"].append(train_metrics["darcy"])
        history["val_total"].append(val_metrics["total_mse"])
        history["val_mask"].append(val_metrics["mask_mse"])
        history["val_nonmask"].append(val_metrics["nonmask_mse"])
        history["val_darcy"].append(val_metrics["darcy"])

        val_loss = val_metrics["total_mse"]

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            torch.save(model.state_dict(), f"{save_prefix}_best_state.pt")

    print(f"Best epoch: {best_epoch}, best val loss: {best_val_loss:.6f}")

    torch.save(model, f"{save_prefix}_final_model.pt")

    return model, history

In [8]:
# model_border_darcy, hist_border_darcy = run_experiment(
#     model_type="splitnet_attn",
#     dataset_mode="border",
#     loss_weights=(1, 1),
#     epochs=250,
#     save_prefix="minimum_info/border_splitattn_darcy.pt"
# )

# model_border_nodarcy, hist_border_nodarcy = run_experiment(
#     model_type="splitnet_attn",
#     dataset_mode="border",
#     loss_weights=(1, 0),
#     epochs=250,
#     save_prefix="minimum_info/border_splitattn_nodarcy.pt"
# )

model_fixed_darcy, hist_fixed_darcy = run_experiment(
    model_type="splitnet_attn",
    dataset_mode="fixed",
    loss_weights=(1, 1),
    epochs=250,
    save_prefix="minimum_info/fixed_splitattn_darcy.pt"
)

model_fixed_nodarcy, hist_fixed_nodarcy = run_experiment(
    model_type="splitnet_attn",
    dataset_mode="fixed",
    loss_weights=(1, 0),
    epochs=250,
    save_prefix="minimum_info/fixed_splitattn_nodarcy.pt"
)

model_random_darcy, hist_random_darcy = run_experiment(
    model_type="splitnet_attn",
    dataset_mode="random",
    loss_weights=(1, 1),
    epochs=250,
    save_prefix="minimum_info/random_splitattn_darcy.pt"
)

model_random_nodarcy, hist_random_nodarcy = run_experiment(
    model_type="splitnet_attn",
    dataset_mode="random",
    loss_weights=(1, 0),
    epochs=250,
    save_prefix="minimum_info/random_splitattn_nodarcy.pt"
)

  0%|          | 0/250 [00:45<?, ?it/s]


KeyboardInterrupt: 